# 4. Geospatial processing

### [read] Summarise data availability of addresses among the fixed_table
- So we know what we're working with in terms of different types

In [1]:
# ### [read] Summarise data availability of addresses among the fixed_table
# - So we know what we're working with in terms of different types
# This should be lat/long, full address, address decomposed across the different fields, just postcode, the nothing
# And if its primary trading address or registered office address
import ibis

address_hierachy = [
    {'primary_trading_address_latitude', 'primary_trading_address_longitude'},
    {'primary_trading_address'},
    {'ro_latitude', 'ro_longitude'},
    {'ro_address'},
    {'ro_address_line_1', 'ro_full_postcode'},
    {'ro_full_postcode'},
    {'ro_postcode'}
]

# Dynamically builds an ibis.cases() expression from a list of column sets.
def build_location_source_cases(table: ibis.expr.types.Table, hierarchy: list[set]):
    cases_list = []
    
    for i, field_set in enumerate(hierarchy, start=1):
        # 1. Build the logical condition: EVERY column in the set must be not-null
        condition = None
        for col_name in field_set:
            is_not_null = table[col_name].notnull()
            condition = is_not_null if condition is None else condition & is_not_null
        
        # 2. Store as a (condition, result_value) tuple
        cases_list.append((condition, i))
        
    # 3. Unpack the list of tuples into ibis.cases. 
    # Fallback MUST be an integer to match the column type.
    fallback_lvl = len(hierarchy) + 1
    return ibis.cases(*cases_list, else_=fallback_lvl)

### [write] Add the ONS postcode directory to the database

In [ ]:
# Add input/NSPL_MAY_2026_UK.csv as a table to the database
# Keep only columns pcds, lat, and long
import ibis
import pandas as pd
from utils.f_0_dirs import get_data_dirs

table_name = "ref_ons_postcode"
file_name = "skinny_NSPL_MAY_2026_UK.csv"
keep_cols = { "pcds", "lat", "long" }

# Initialize connection
dirs = get_data_dirs(segment="descriptives")
con = ibis.duckdb.connect(str(dirs.db_path))

csv_file = f"{dirs.input_dir}/{file_name}"

df_raw = pd.read_csv(csv_file, usecols=list(keep_cols))
# df_raw.to_csv(f"{dirs.input_dir}/skinny_{file_name}")
if len(set(df_raw.columns).intersection(keep_cols)) != len(keep_cols):
    for col in keep_cols:
        if col not in set(df_raw.columns):            
            raise ValueError(f"Expected {col} in columns: missing")

codes_length = len(df_raw)
display(df_raw.sample(20))

❌ DATA_DIR path does not exist or is not a directory: /mnt/h/Other computers/My computer/fame_clean/1_FAME_raw_data/2025.07.30


In [3]:
con.create_table(table_name, df_raw, overwrite=table_name in con.list_tables())
table_ref = con.table(table_name)
rows_count = table_ref.count().execute()
print(f"{rows_count:,} rows in new table {table_name}")
print("Sample:")
display(table_ref.sample(20 / rows_count).execute())

2,726,477 rows in new table ref_ons_postcode
Sample:


,pcds,lat,long
0,BH5 1JP,50.722251,-1.835667
1,BT20 4WU,99.999999,0.000000
2,EX6 8PL,50.628684,-3.451323
3,HR4 9UU,52.071887,-2.721395
4,KT20 6NW,51.285990,-0.214215
5,L4 5SH,53.441079,-2.965222
6,LL13 7EY,53.036467,-2.991550
7,ME2 1BQ,51.352562,0.442107
8,NP23 5TP,51.798839,-3.225913
9,PL22 0AR,50.409628,-4.673098


### [write] Create new column for address geocoding

In [ ]:
import ibis
from utils.f_0_dirs import get_data_dirs
from f_3_spatial import convert_dms_to_decimal

old_table_name = "fame_fixed_filtered"
new_table_name = "working_fixed"

# Initialize connection
dirs = get_data_dirs()
con = ibis.duckdb.connect(str(dirs.db_path))
con.raw_sql("INSTALL spatial; LOAD spatial;")

# Reference the existing tables
fame_fixed = con.table(old_table_name)

# 1. Parse availability into an indicator and extract the best available text address
sep: ibis.StringScalar = ibis.literal(", ", type="string")
working_raw = fame_fixed.mutate(
    address_raw_lvl = build_location_source_cases(fame_fixed, address_hierachy),
    address_raw = ibis.coalesce(
        fame_fixed.primary_trading_address,
        fame_fixed.ro_address,
        # Fallback: concatenate the separate lines and postcode if above are null
        sep.join(
            ibis.array([
                fame_fixed.ro_address_line_1, 
                fame_fixed.ro_address_line_2, 
                fame_fixed.ro_full_postcode
            ]).filter(lambda x: x.notnull())
        ),
        fame_fixed.ro_full_postcode,
        fame_fixed.ro_postcode
    )
)

# 2. Extract and clean postcodes for Levels 2, 4, 5, 6, and 7
# Regex matches standard UK postcode formats. We strip whitespace and uppercase for perfect joins.
# 2. Extract and clean postcodes for Levels 2, 4, 5, 6, and 7
# Regex matches standard UK postcode formats. 
uk_postcode_regex = r"([A-Za-z]{1,2}\d[A-Za-z\d]?\s?\d[A-Za-z]{2})"

working_pc = working_raw.mutate(
    extracted_postcode = ibis.cases(
        (working_raw.address_raw_lvl.isin([1, 2]), working_raw.primary_trading_address.re_extract(uk_postcode_regex, 1)),
        (working_raw.address_raw_lvl.isin([3, 4]), working_raw.ro_address.re_extract(uk_postcode_regex, 1)),
        (working_raw.address_raw_lvl.isin([5, 6]), working_raw.ro_full_postcode),
        (working_raw.address_raw_lvl == 7, working_raw.ro_postcode),
        else_=ibis.literal(None, type="string")
    )
    .cast("string")
    .upper()
    .re_replace(r"\s+", "") # Step 1: Strip all existing spaces (e.g. "EC1Y2AL" or "EC1Y  2AL" -> "EC1Y2AL")
    .re_replace(r"(.+)(\d[A-Z]{2})$", r"\1 \2") # Step 2: Insert exactly one space before the 3-character inward code
)

# Clean the ONS directory postcodes using the same logic for the join
ons_table_name = "ref_ons_postcode" # Assume this exists with 'postcode', 'lat', 'long'
ons_lookup = con.table(ons_table_name) 

# 3. Join firm data with the ONS lookup
working_joined = working_pc.left_join(
    ons_lookup,
    working_pc.extracted_postcode == ons_lookup.pcds
)

# 4. Extract standardized spatial coordinates based on ALL 7 levels of the hierarchy
working_fixed = working_joined.mutate(
    address_case = ibis.cases(
        (working_joined.address_raw_lvl <= 2, ibis.literal('pta')),
        (working_joined.address_raw_lvl <= 7, ibis.literal('ro')),
        else_=ibis.literal(None, type="string")
    ),
    address_lvl = ibis.cases(
        (working_joined.address_raw_lvl <= 2, working_joined.address_raw_lvl),
        (working_joined.address_raw_lvl <= 7, working_joined.address_raw_lvl - 2),
        else_=ibis.literal(None, type="int64")
    ),
    lat_dec = ibis.cases(
        (working_joined.address_raw_lvl == 1, convert_dms_to_decimal(working_joined.primary_trading_address_latitude)),
        (working_joined.address_raw_lvl == 3, convert_dms_to_decimal(working_joined.ro_latitude)),
        else_=working_joined.lat # ibis.literal(None, type='float64') # # Automatically covers levels 2, 4, 5, 6, and 7
    ),
    lon_dec = ibis.cases(
        (working_joined.address_raw_lvl == 1, convert_dms_to_decimal(working_joined.primary_trading_address_longitude)),
        (working_joined.address_raw_lvl == 3, convert_dms_to_decimal(working_joined.ro_longitude)),
        else_=working_joined.long # ibis.literal(None, type='float64') # # Automatically covers levels 2, 4, 5, 6, and 7
    )
)

working_fixed_loc = working_fixed.select('registered_number', 'address_raw', 'extracted_postcode',
         'address_case', 'address_lvl', 'lat_dec', 'lon_dec')
# ).drop("extracted_postcode", "postcode", "lat", "long") # Drop join artifacts

row_count = working_fixed_loc.count().execute()
display(working_fixed_loc.sample(30 / row_count).execute())

❌ DATA_DIR path does not exist or is not a directory: /mnt/h/Other computers/My computer/fame_clean/1_FAME_raw_data/2025.07.30


,registered_number,address_raw,extracted_postcode,address_case,address_lvl,lat_dec,lon_dec
0,07308880,"Lancaster House, Drayton Road, Shirley, Solihu...",B90 4NG,pta,1,52.398861,-1.802889
1,12744628,"The Old Grange, Warren Estate, Lordship Road, ...",CM1 3WT,ro,1,51.736139,0.429139
2,04361700,"1110 Elliott Court, Coventry Business Park, Co...",CV5 6UB,ro,1,52.403194,-1.553972
3,13728881,"54 Cubbington Road, Coventry, West Midlands, C...",CV6 7BN,pta,2,52.442847,-1.483345
4,07051686,"c/o Portal House, Unit 2, Botterley Court, Nan...",CW6 9GT,pta,2,53.117982,-2.598863
5,SC130376,"Suite 9 River Court, 5 West Victoria Dock Road...",DD1 3JT,ro,2,56.459560,-2.961632
6,10130324,"Unit 9 Victory Park, Victory Road, Derby, Derb...",DE24 8ZF,ro,1,52.892750,-1.471167
7,06427917,"Satago Cottage, 360A Brighton Road, Croydon, S...",CR2 6AL,ro,1,51.350000,-0.100167
8,SC237325,"Capital Square, 58 Morrison Street, Edinburgh,...",EH3 8EP,ro,2,55.947945,-3.192256
9,SC191533,"210 Edmiston Drive, Glasgow, Lanarkshire, G51 2YU",G51 2YU,ro,1,55.853167,-4.314111


In [7]:
working_fixed_skinny = working_fixed    \
    .mutate(
        is_public = working_fixed.ticker_symbol.notnull()

    # fame_fixed variables which we want to keep for the regression
    ).select(
        'registered_number', 'company_name', 'is_public', 'industry_codes', 'file_codes',
        'primary_uk_sic_2007_code', 'primary_uk_sic_2007_description',

    # new location variables
        'lat_dec', 'lon_dec', 'address_lvl', 'address_case'
    )

print(f"✅ Inserting columns into new '{new_table_name}' table.")
con.create_table(new_table_name, working_fixed_skinny, overwrite=True)

# Verify the final materialized table
final_table = con.table(new_table_name)
row_count = final_table.count().execute()
col_count = len(final_table.columns)

print(f"✅ Materialized '{new_table_name}' table.")
print(f"📊 Number of rows: {row_count:,}")
print(f"📊 Number of columns: {col_count}")
print(f"\nSample of {new_table_name}:")
display(final_table.sample(30 / row_count).execute())

✅ Inserting columns into new 'working_fixed' table.
✅ Materialized 'working_fixed' table.
📊 Number of rows: 152,379
📊 Number of columns: 11

Sample of working_fixed:


,registered_number,company_name,is_public,industry_codes,file_codes,primary_uk_sic_2007_code,primary_uk_sic_2007_description,lat_dec,lon_dec,address_lvl,address_case
0,03671019,WHITES ENGINEERING LIMITED,False,30,22_37 1,30990,Manufacture of other transport equipment n.e.c.,52.761556,-0.643833,1,pta
1,12432902,RICHARDSON CARE HOLDINGS LIMITED,False,87,19_22,87900,Other residential care activities,52.261488,-0.898416,2,pta
2,11974148,PST HOLDINGS LIMITED,False,64,12_47,64209,Activities of other holding companies (not inc...,51.167942,-2.698118,2,ro
3,10746789,WEBCARE GROUP LIMITED,False,64,12_47,64209,Activities of other holding companies (not inc...,53.592111,-2.420139,1,ro
4,03887202,B W EUROPE LIMITED,False,82,10_14 2,82990,Other business support service activities n.e.c.,51.415972,-0.755611,1,ro
5,01819517,AUTOSTART (MIDLANDS) LIMITED,False,71,18_27,71129,Other engineering activities (not including en...,52.565500,-1.820889,1,ro
6,07237305,OASIS COMMUNITY HUB: WATERLOO,False,"93,88","17_15,16_50",88990,Other social work activities without accommoda...,51.497056,-0.111722,1,pta
7,02592883,BAP GROUP LIMITED,False,52,17_05,52103,Operation of warehousing and storage facilitie...,51.888185,0.894119,2,ro
8,05459368,GWP GROUP LIMITED,False,17,22_05,17219,Manufacture of paper and paperboard containers...,52.364151,-1.468077,2,ro
9,09827436,8 SLICES LIMITED,False,56,17_22,56101,Licensed restaurants,52.483750,-1.281472,1,ro
